# Missing values and honest cleaning · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 10. Missing values and honest cleaning

Find missing entries and choose a treatment that fits the question.

Before you start: You can convert columns and use Boolean masks.

Official reference: https://pandas.pydata.org/docs/user_guide/missing_data.html

### M10C01. Missing is not the same as empty or zero

`isna()` gives True where a value is missing. Zero is a recorded value, and an empty string is text unless you explicitly treat it as missing. Counting True values with `.sum()` counts missing entries.

**Read:** What type of values does `isna()` return?

Answer: True/False values, one for each entry.

**Predict:** How many missing entries will be counted?

Answer: Three: None, pd.NA, and np.nan.

**Explain:** Why is the empty string not automatically missing?

Answer: It is a stored text value. A cleaning rule must say whether blank text represents missing data.



In [ ]:
# Predict before running: How many missing entries will be counted?
values = pd.Series([0, None, "", pd.NA, np.nan], dtype="object")
result = values.isna()
print(result.tolist())
print("Missing count:", result.sum())

**Your turn:** Treat the empty string as missing, but keep zero. Store the cleaned Series in `result` and count missing entries.

For the check, keep these result names: `result`.

**Hint:** Use `.replace("", pd.NA)`.

**Watch for this:** Use `.isna()`, not `== np.nan`, to detect missing values.

In [ ]:
# Change this code to complete the task.
# Question: Treat the empty string as missing, but keep zero. Store the cleaned Series in `result` and count missing entries.
values = pd.Series([0, None, "", pd.NA, np.nan], dtype="object")
result = values.replace("", pd.NA)
print(result)
print("Missing count:", result.isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.isna().sum() == 4
assert result.iloc[0] == 0
print("Code checks passed. Review the written explanation as well.")

Four values are now missing. The recorded zero is still zero.

**Quick check:** Which original value is known rather than missing?

1. None
2. 0
3. np.nan

Answer: 2. Zero is a numeric value; it should not disappear just because some data are missing.

**Explain what you learned:** Define what missing means in your dataset.

### M10C02. Drop rows only for required fields

`dropna(subset=[...])` removes rows missing the specified fields. It does not remove a row just because an unrelated field is missing. Keep a count of excluded records and consider how the exclusion affects the answer.

**Read:** Which column controls row removal in this cell?

Answer: Score only.

**Predict:** Will S006 remain even though Study_Hours is missing?

Answer: Yes. S006 has a score, so it passes the Score-only requirement.

**Explain:** Why not use `df.dropna()` without thinking?

Answer: It may remove rows for fields the analysis does not need, changing the sample unnecessarily.



In [ ]:
# Predict before running: Will S006 remain even though Study_Hours is missing?
df = raw_students()
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
result = df.dropna(subset=["Score"])
print(result[["Student_ID", "Score", "Study_Hours"]])
print("Rows kept:", len(result))

**Your turn:** Keep only rows with both Score and Study_Hours available. Leave the intentional duplicate for the next module.

For the check, keep these result names: `result`.

**Hint:** Pass both required column names to `subset`.

**Watch for this:** Removing incomplete records can change the composition of your data; report what was excluded.

In [ ]:
# Change this code to complete the task.
# Question: Keep only rows with both Score and Study_Hours available. Leave the intentional duplicate for the next module.
df = raw_students()
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
result = df.dropna(subset=["Score", "Study_Hours"])
print(result[["Student_ID", "Score", "Study_Hours"]])
print("Rows kept:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 10
assert not set(["S004", "S006", "S008"]) & set(result["Student_ID"])
print("Code checks passed. Review the written explanation as well.")

Ten rows remain out of thirteen. S004, S008, and S006 were excluded for the stated requirements.

**Quick check:** What does subset specify?

1. The fields that must be present
2. The first few rows to show
3. A replacement for missing values

Answer: 1. Only missing values in those columns trigger removal.

**Explain what you learned:** Match required fields to the analysis question.

### M10C03. Fill a label without inventing a measurement

`fillna()` replaces missing values with a value you specify. A label such as "Unknown" can make a missing category visible in a report. The same idea does not justify inventing a score or age.

**Read:** What is being filled here: a measured number or a category label?

Answer: A category label.

**Predict:** How many Unknown labels will appear?

Answer: Two.

**Explain:** Does Unknown tell us the actual campus?

Answer: No. It makes the missing information explicit; it does not recover the campus.



In [ ]:
# Predict before running: How many Unknown labels will appear?
campuses = pd.Series(["RAK", None, "Dubai", None], dtype="string")
result = campuses.fillna("Unknown")
print(result.tolist())

**Your turn:** Use the same Series, but first record a Boolean Was_missing flag. Store a DataFrame named result with the filled Campus column and the Was_missing flag.

For the check, keep these result names: `result`.

**Hint:** Call `.isna()` before replacing missing values.

**Watch for this:** Do not use zero as a universal replacement for missing numbers.

In [ ]:
# Change this code to complete the task.
# Question: Use the same Series, but first record a Boolean Was_missing flag. Store a DataFrame named result with the filled Campus column and the Was_missing flag.
campuses = pd.Series(["RAK", None, "Dubai", None], dtype="string")
result = pd.DataFrame({"Campus": campuses.fillna("Unknown"), "Was_missing": campuses.isna()})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Was_missing"].sum() == 2
assert (result.loc[result["Was_missing"], "Campus"] == "Unknown").all()
print("Code checks passed. Review the written explanation as well.")

The flag preserves which labels were originally missing, even after every cell contains text.

**Quick check:** Why retain a missingness flag?

1. To restore information about which entries were filled
2. To make every row identical
3. To prove the missing values were incorrect

Answer: 1. After filling, the flag distinguishes observed labels from replacements.

**Explain what you learned:** Keep missingness visible when replacements are used.

### M10C04. Compare a fill policy with keeping missing data

Preview of Module 16: the groupby, transform, and named aggregation syntax in this cell will be explained there. First focus on the difference between a technical replacement and a justified data policy. A group median is one possible imputation rule, not a universal answer. `transform()` returns a group result for every original row. This small example compares a proposed fill with the original values so its effect is visible.

**Read:** Why does transform return five values instead of two group summaries?

Answer: It repeats each group median on the rows belonging to that group.

**Predict:** Which values fill the missing AI and IT hours?

Answer: AI gets 6.0, the median of 4 and 8; IT gets 2.0, its only observed value.

**Explain:** Why is filling the IT value with 2.0 a weakly supported choice?

Answer: It is based on only one observed IT value and can hide uncertainty or reduce apparent variation.



In [ ]:
# Predict before running: Which values fill the missing AI and IT hours?
df = pd.DataFrame({"Program": ["AI", "AI", "AI", "IT", "IT"], "Hours": [4.0, 8.0, np.nan, 2.0, np.nan]})
group_median = df.groupby("Program")["Hours"].transform("median")
result = df.assign(Hours_filled=df["Hours"].fillna(group_median), Was_missing=df["Hours"].isna())
print(result)

**Your turn:** Use the same original table. Keep the missing values and store a summary in result with Program as its index. Name the known Hours count Known_hours, the row count Rows, and the median Median_hours.

For the check, keep these result names: `df`, `result`.

**Hint:** Use named aggregation; count counts known Hours, while size counts rows.

**Watch for this:** In predictive modeling, fit any imputation rule on training data only; do not learn it from held-out test data.

In [ ]:
# Change this code to complete the task.
# Question: Use the same original table. Keep the missing values and store a summary in result with Program as its index. Name the known Hours count Known_hours, the row count Rows, and the median Median_hours.
df = pd.DataFrame({"Program": ["AI", "AI", "AI", "IT", "IT"], "Hours": [4.0, 8.0, np.nan, 2.0, np.nan]})
result = df.groupby("Program").agg(Known_hours=("Hours", "count"), Rows=("Hours", "size"), Median_hours=("Hours", "median"))
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc["AI", "Known_hours"] == 2
assert result.loc["IT", "Rows"] == 2
assert df["Hours"].isna().sum() == 2
print("Code checks passed. Review the written explanation as well.")

AI has 2 known values in 3 rows; IT has 1 in 2. The report shows the missing information rather than replacing it.

**Quick check:** Should every missing measurement be filled with a mean or median?

1. Yes, always
2. Only when the policy fits the context and its effects are checked
3. Only if the column is first converted to text

Answer: 2. Choose and document a treatment; retaining missing values or excluding specific rows may be more appropriate.

**Explain what you learned:** Explain the assumption behind every fill policy.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates